# Benchmark Contributions Tutorial

This tutorial shows how to contribute benchmark results, view leaderboards, and compare against reference baselines.

In [ ]:
# Install if needed
!pip install -e . --quiet

## Overview

The benchmark contribution system provides:
- Standard tasks with reference implementations
- Reference results (baselines) for comparison
- Community contribution submission
- Automated verification against references
- Leaderboards for each task

## Available Tasks

In [ ]:
# List available benchmark tasks
!comp benchmark-contrib list-tasks

## Viewing Task Details

In [ ]:
# Show MNIST classification task details
!comp benchmark-contrib show-task mnist_classification

## Viewing Reference Results

In [ ]:
# List reference results for MNIST
!comp benchmark-contrib list-references mnist_classification

## Viewing Leaderboards

In [ ]:
# Show leaderboard for MNIST (default: validation_accuracy)
!comp benchmark-contrib leaderboard mnist_classification

In [ ]:
# Show leaderboard for CIFAR-10
!comp benchmark-contrib leaderboard cifar10_classification

In [ ]:
# Show leaderboard for Tiny Shakespeare (lower validation_loss is better)
!comp benchmark-contrib leaderboard tiny_shakespeare_lm --metric validation_loss --lower-better

## Submitting a Contribution

After running your own experiments, you can submit results:

In [ ]:
# Submit a contribution (example - replace with your actual results)
# !comp benchmark-contrib submit mnist_classification my_new_model \
#     --model-config '{"hidden_dims": [512, 256], "activation": "gelu"}' \
#     --metrics '{"validation_accuracy": 0.987, "validation_loss": 0.038}' \
#     --contributor "your_name" \
#     --seeds 5 \
#     --seed-values 42 123 456 789 999 \
#     --hardware "NVIDIA A100" \
#     --runtime 25.3

## Verifying Contributions

Contributions are automatically verified against reference baselines:

In [ ]:
# Verify all contributions for a task
!comp benchmark-contrib verify mnist_classification

## Listing Contributions

In [ ]:
# List all contributions for MNIST
!comp benchmark-contrib list-contributions mnist_classification

## Exporting Data

Export all benchmark data for external analysis:

In [ ]:
# Export all data to JSON
!comp benchmark-contrib export benchmark_data.json

## Python API

You can also use the Python API directly:

In [ ]:
from computronium.benchmarks.contributions import (
    contribution_registry,
    leaderboard,
    BenchmarkContribution,
)
from datetime import datetime

# List tasks programmatically
tasks = contribution_registry.list_tasks()
for t in tasks:
    print(f"{t.task_id}: {t.name} ({t.domain}, {t.difficulty})")

In [ ]:
# Build leaderboard programmatically
lb = leaderboard(
    "mnist_classification",
    primary_metric="validation_accuracy",
    higher_is_better=True,
    registry=contribution_registry,
)
entries = lb.build()
print(lb.to_table(entries))

In [ ]:
# Submit a contribution programmatically
contrib = BenchmarkContribution(
    contribution_id=f"mnist_classification_my_experiment_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    task_id="mnist_classification",
    model_name="my_experiment",
    model_config={"hidden_dims": [1024, 512], "activation": "silu"},
    metrics={"validation_accuracy": 0.986, "validation_loss": 0.042},
    n_seeds=5,
    seed_values=(42, 123, 456, 789, 999),
    contributor="researcher_name",
    hardware="NVIDIA H100",
    runtime_seconds=30.5,
)

# Compare against reference
comparison = contribution_registry.compare_against_reference(
    contrib.task_id, contrib.model_name, contrib.metrics
)
print(f"Comparison: {comparison}")

# Add to registry
contrib = BenchmarkContribution(
    contribution_id=contrib.contribution_id,
    task_id=contrib.task_id,
    model_name=contrib.model_name,
    model_config=contrib.model_config,
    metrics=contrib.metrics,
    reference_comparison=comparison,
    n_seeds=contrib.n_seeds,
    seed_values=contrib.seed_values,
    contributor=contrib.contributor,
    hardware=contrib.hardware,
    runtime_seconds=contrib.runtime_seconds,
)
contribution_registry.add_contribution(contrib)
print("Contribution added!")

In [ ]:
# Verify programmatically
contribs = contribution_registry.get_contributions("mnist_classification")
task = contribution_registry.get_task("mnist_classification")

for c in contribs:
    if not c.verified:
        passed = True
        if task.min_performance:
            for metric, min_val in task.min_performance.items():
                if c.metrics.get(metric, 0) < min_val:
                    passed = False
                    print(f"FAIL: {metric} below minimum")

        if c.reference_comparison:
            for metric, diff in c.reference_comparison.items():
                lower_is_better = any(
                    k in metric.lower()
                    for k in [
                        "loss",
                        "error",
                        "time",
                        "latency",
                        "energy",
                        "flops",
                        "memory",
                    ]
                )
                if lower_is_better and diff > 0.05:
                    passed = False
                    print(f"FAIL: {metric} regressed")
                elif not lower_is_better and diff < -0.05:
                    passed = False
                    print(f"FAIL: {metric} regressed")

        if passed:
            print(f"PASS: {c.model_name} verified")

## Next Steps

- Run your own experiments and submit results
- Check out the [quickstart](quickstart.ipynb) for running experiments
- Learn about [campaigns](campaigns.ipynb) for systematic comparisons
- Explore [joint system composition](joint_system_composition.ipynb)